# 04 — Activation patching (the headline experiment)

For each (layer, head) replace its `z` from the safe run into the harm run. Heatmap the Δ.


In [ ]:
from safety_circuits.config import MODELS, REPO_ROOT, RESULTS_DIR
from safety_circuits.models import load_model
from safety_circuits.patching import patch_each_head, patch_residual_stream
from safety_circuits.analysis import aggregate_pairs, head_heatmap, plot_heatmap
from safety_circuits.data import load_jsonl


In [ ]:
loaded = load_model(MODELS['qwen2.5'])
pairs = load_jsonl(REPO_ROOT / 'data' / 'processed' / 'pairs.jsonl')[:8]  # start tiny


### Coarse trace first — which *layer band* matters?


In [ ]:
r0 = patch_residual_stream(loaded, pairs[0]['harm']['text'], pairs[0]['safe']['text'])
import pandas as pd
pd.DataFrame([x.__dict__ for x in r0]).set_index('layer')['delta_margin'].plot.bar()


### Now the fine head sweep — one pair at a time, aggregate.


In [ ]:
per_pair = []
for r in pairs:
    per_pair.append(patch_each_head(loaded, r['harm']['text'], r['safe']['text']))
agg = aggregate_pairs(per_pair)
agg.head(15)


In [ ]:
grid = head_heatmap(agg, n_layers=loaded.n_layers, n_heads=loaded.n_heads)
RESULTS_DIR.mkdir(exist_ok=True)
plot_heatmap(grid, title='TinyLlama: per-head |Δ refusal-margin|',
             save_to=str(RESULTS_DIR / 'qwen2.5_heatmap.png'))


In [ ]:
agg.to_csv(RESULTS_DIR / 'qwen2.5_patch_z.csv', index=False)
print('Top candidates:')
print(agg[agg.component == 'z'].head(10))
